# PersonaPlex 4-bit NF4 Quantization
**Goal:** Memory reduction via bitsandbytes NF4 (fp16 ~14GB → ~3.5–4GB)

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q bitsandbytes>=0.43.0
!pip show bitsandbytes | grep Version

Version: 0.49.2


In [ ]:
# ── Cell 2: Imports & path config ─────────────────────────────────────────
import sys
sys.path.insert(0, '/content/src/moshi-personaplex/moshi')

import os
import torch
import bitsandbytes as bnb
from bitsandbytes.nn import Linear4bit
from moshi.models import loaders

MODEL_PATH  = "/root/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/31c4ae40e3710304737e6f7b70df23bd06917e5c"
OUTPUT_PATH = "/content/personaplex_4bit_nf4.pt"
DEVICE      = "cuda"

print(f"bitsandbytes : {bnb.__version__}")
print(f"torch        : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

ModuleNotFoundError: No module named 'moshi'

In [ ]:
# ── Cell 3: Approach A — try HF BitsAndBytesConfig path ───────────────────
# get_moshi_lm() agar internally from_pretrained use karta hai toh
# quantization_config directly accept ho jaayegi — sabse clean path.

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — LLMs ke liye best
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,      # nested quant: ~0.4 bits/param extra save
)

lm = None
try:
    print("Attempting HF-style load with BitsAndBytesConfig...")
    lm = loaders.get_moshi_lm(
        f"{MODEL_PATH}/model.safetensors",
        device=DEVICE,
        quantization_config=bnb_config,
    )
    print("✅ HF-style load succeeded!")
except TypeError as e:
    print(f"⚠️  Loader does not accept quantization_config: {e}")
    print("→ Will use manual layer replacement in Cell 5.")

In [ ]:
# ── Cell 4: Approach B (fallback) — load in fp16 first ────────────────────
# Sirf tab run karo jab Cell 3 mein lm = None raha ho.

if lm is None:
    print("Loading model in bf16 before manual quantization...")
    lm = loaders.get_moshi_lm(
        f"{MODEL_PATH}/model.safetensors",
        device=DEVICE,
    )
    print("✅ Model loaded!")
else:
    print("Cell 3 succeeded — skipping fp16 load.")

In [ ]:
# ── Cell 5: Approach B (fallback) — manual layer replacement ──────────────
# Har nn.Linear ko Linear4bit (NF4) se replace karta hai.
# LM head skip kiya hai — usse quantize karne se logit quality drop hoti hai.

HEAD_KEYWORDS = ["lm_head", "output_proj", "head"]

def replace_linear_with_4bit(model: torch.nn.Module) -> torch.nn.Module:
    replaced = 0

    def _replace(parent, prefix=""):
        nonlocal replaced
        for name, child in list(parent.named_children()):
            full_name = f"{prefix}.{name}" if prefix else name

            if isinstance(child, torch.nn.Linear):
                if any(kw in full_name for kw in HEAD_KEYWORDS):
                    print(f"  [skip] {full_name}")
                    continue

                new_layer = Linear4bit(
                    child.in_features,
                    child.out_features,
                    bias=child.bias is not None,
                    quant_type="nf4",
                    compute_dtype=torch.bfloat16,
                )
                new_layer.weight = bnb.nn.Params4bit(
                    child.weight.data,
                    requires_grad=False,
                    quant_type="nf4",
                )
                if child.bias is not None:
                    new_layer.bias = child.bias

                setattr(parent, name, new_layer)
                replaced += 1
            else:
                _replace(child, full_name)

    if lm is not None:
        print("Replacing nn.Linear → Linear4bit (NF4)...")
        _replace(lm)
        print(f"✅ Done! {replaced} layers quantized.")
    else:
        print("⚠️  lm is None — Cell 4 pehle run karo.")

In [ ]:
# ── Cell 6: Run replacement (only if Approach B) ──────────────────────────
# Cell 3 fail hua tha toh hi yeh run karo.

replace_linear_with_4bit(lm)

In [ ]:
# ── Cell 7: GPU memory check ───────────────────────────────────────────────

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    print(f"GPU Allocated : {allocated:.2f} GB")
    print(f"GPU Reserved  : {reserved:.2f} GB")
else:
    print("CUDA not available")

In [ ]:
# ── Cell 8: Save quantized model ──────────────────────────────────────────
# NOTE: Reload ke waqt bitsandbytes installed hona zaroori hai
#       kyunki weights Params4bit format mein hain.

print(f"Saving to {OUTPUT_PATH} ...")
torch.save(lm.state_dict(), OUTPUT_PATH)

size_gb = os.path.getsize(OUTPUT_PATH) / 1024**3
print(f"✅ Saved! File size: {size_gb:.2f} GB")
print()
print("Expected sizes:")
print("  fp16 original  → ~14.0 GB")
print("  NF4 + dq       →  ~3.5–4.0 GB")